# **RANDOM FOREST**

**MODEL 2**



**Recreate X_train/X_val at the top of Notebook 3**

In [1]:
# Notebook 3 — Cell 0: Load prepared data from Notebook 2

import pandas as pd
import joblib

# Load the modeling table again
df = pd.read_csv("data/modeling_table.csv", parse_dates=['date'])

# Re-run the same preprocessing steps from Notebook 2
# (leakage removal, stage-5 policy, train/val split, dropna, one-hot encode season)

# 1. Remove leakage
leakage_cols = ['latent_demand', 'markdown_depth', 'days_at_current_price']
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])

# 2. Stage-5 policy columns
df['sample_weight'] = 1.0
df['downsample_prob'] = 1.0

# 3. Train/val split
cutoff = df['date'].max() - pd.Timedelta(days=60)
train_df = df[df['date'] <= cutoff].copy().reset_index(drop=True)
val_df   = df[df['date'] >  cutoff].copy().reset_index(drop=True)

# 4. Drop NaNs
train_df = train_df.dropna().reset_index(drop=True)
val_df   = val_df.dropna().reset_index(drop=True)

# 5. One-hot encode season consistently
combined = pd.concat([train_df, val_df], axis=0)
combined = pd.get_dummies(combined, columns=['season'], drop_first=False)

train_df = combined.iloc[:len(train_df)].reset_index(drop=True)
val_df   = combined.iloc[len(train_df):].reset_index(drop=True)

# 6. Build FEATURES and X/y
TARGET = "units_sold"
excluded_cols = {'sku_id', 'date', TARGET, 'downsample_prob'}

FEATURES = [c for c in train_df.columns if c not in excluded_cols and c != 'sample_weight']

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].values

X_val   = val_df[FEATURES].copy()
y_val   = val_df[TARGET].values

sample_weight_train = train_df['sample_weight'].values

print("Notebook 3 setup complete.")
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)


Notebook 3 setup complete.
X_train shape: (234200, 24)
X_val shape: (20880, 24)


In [2]:
# Notebook 3 — Cell 1: Imports and Random Forest baseline

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# sanity checks (you should have these from Notebook 2)
assert 'X_train' in globals() and 'X_val' in globals(), "Run Notebook 2 to prepare X/y first."
assert len(X_train) > 0 and len(X_val) > 0, "Empty train/val — check previous split."

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rmsle(y_true, y_pred):
    y_pred_nonneg = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_nonneg)))

# Random Forest baseline config (you can tune later)
RF_PARAMS = {
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "n_jobs": -1,
    "random_state": 123,
}

rf = RandomForestRegressor(**RF_PARAMS)

# fit with sample weights (same as LR)
rf.fit(X_train, y_train, sample_weight=sample_weight_train)

# predict on validation
y_pred_rf = rf.predict(X_val)
y_pred_rf_clipped = np.clip(y_pred_rf, 0, None)

print("Random Forest — validation metrics (clipped preds):")
print("  RMSE :", round(rmse(y_val, y_pred_rf_clipped), 4))
print("  MAE  :", round(mean_absolute_error(y_val, y_pred_rf_clipped), 4))
print("  RMSLE:", round(rmsle(y_val, y_pred_rf_clipped), 4))
print("  mean_true:", round(np.mean(y_val), 4),
      "mean_pred:", round(np.mean(y_pred_rf_clipped), 4))


Random Forest — validation metrics (clipped preds):
  RMSE : 0.0601
  MAE  : 0.024
  RMSLE: 0.0496
  mean_true: 0.0003 mean_pred: 0.0238


# Random Forest Feature Importances


We will understand **WHY** RF performs well

In [3]:
# Notebook 3 — Cell 2: Random Forest Feature Importances

import pandas as pd
import numpy as np

# Extract feature importances
importances = rf.feature_importances_
feature_names = FEATURES

# Build a DataFrame
fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

# Sort descending
fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 15 Random Forest Feature Importances:")
print(fi_df.head(15))

# Save to CSV for later comparison with XGBoost
fi_path = "results/feature_importances_rf.csv"
fi_df.to_csv(fi_path, index=False)
print(f"Saved RF feature importances to {fi_path}")


Top 15 Random Forest Feature Importances:
               feature  importance
0       rolling_7d_avg    0.631129
1      inventory_ratio    0.073675
2     lag_7_units_sold    0.052001
3   days_to_season_end    0.045716
4    days_since_launch    0.044998
5    lag_14_units_sold    0.040085
6           is_holiday    0.015743
7         discount_pct    0.014604
8       markdown_stage    0.014566
9         dow_Saturday    0.010463
10    category_jackets    0.008112
11          dow_Sunday    0.007182
12      category_jeans    0.005544
13      category_shoes    0.004832
14          dow_Friday    0.004183
Saved RF feature importances to results/feature_importances_rf.csv


# **📘 Random Forest Feature Importances — Summary & Interpretation**


# Random Forest delivered excellent validation performance (RMSE ≈ 0.06, RMSLE ≈ 0.049).
# To understand why the model performs so well, we examined its feature importances — a measure of how much each feature contributed to reducing prediction error.

# Feature importance helps us interpret the model’s behavior, identify the strongest predictors, and understand the underlying demand patterns in our retail dataset.

**Key Takeaways**

rolling_7d_avg (63%) — The strongest predictor. Recent demand is the best signal of future demand.

inventory_ratio (7%) — Demand is constrained by stock availability.

lag_7_units_sold + lag_14_units_sold (~9%) — Weekly and bi‑weekly demand memory.

days_to_season_end + days_since_launch (~9%) — Product lifecycle and seasonality.

discount_pct + markdown_stage (~3%) — Price signals matter but are secondary.

dow_ + is_holiday (~3%)* — Calendar effects add nuance.

category_ features (~1–2%)* — Category matters, but less than temporal and inventory signals.

**Why This Matters**

These importances show that Random Forest is learning realistic retail behavior:

demand momentum

inventory constraints

lifecycle curves

holiday/weekend effects

price sensitivity

This confirms the model’s strong performance and guides what to improve or tune next.

# **RF Error Analysis**


This cell helps me to see **where RF makes mistakes**, which SKUs are hardest, and whether errors cluster around certaind dates or categories

In [4]:
import pandas as pd
import numpy as np


#now let us build dataframe for error
error_df = pd.DataFrame({
    "sku_id": val_df["sku_id"],
    "date": val_df["date"],
    "true_units_sold": y_val,
    "pred_units_sold": y_pred_rf_clipped
})

error_df["abs_error"] = np.abs(error_df["true_units_sold"] - error_df["pred_units_sold"])

# Sort by largest errors
top_errors = error_df.sort_values("abs_error", ascending=False).head(20)

print("Top 20 RF Errors:")
print(top_errors)

# Basic error distribution stats
print("\nError Distribution Summary:")
print(error_df["abs_error"].describe())



Top 20 RF Errors:
         sku_id       date  true_units_sold  pred_units_sold  abs_error
15268  SKU_0364 2023-11-30                1         0.000000   1.000000
20674  SKU_0496 2023-12-06                1         0.038250   0.961750
19807  SKU_0480 2023-11-09                1         0.093056   0.906944
11796  SKU_0272 2023-12-08                0         0.861714   0.861714
11810  SKU_0272 2023-12-22                0         0.861714   0.861714
11817  SKU_0272 2023-12-29                0         0.861714   0.861714
11803  SKU_0272 2023-12-15                0         0.861714   0.861714
11789  SKU_0272 2023-12-01                0         0.859214   0.859214
20828  SKU_0499 2023-11-10                1         0.172722   0.827278
20836  SKU_0499 2023-11-18                1         0.304389   0.695611
11732  SKU_0271 2023-12-04                0         0.632631   0.632631
11799  SKU_0272 2023-12-11                0         0.632631   0.632631
11806  SKU_0272 2023-12-18                0   

# 📘 Random Forest Error Analysis — Summary
Random Forest achieves very strong performance overall (mean absolute error ≈ 0.024), but the Top 20 errors reveal two predictable patterns:

**1. Under‑prediction of rare positive sales (true = 1, pred ≈ 0.0–0.3)**
These cases occur when a SKU sells a single unit after a long period of zero demand. RF tends to predict small positive values instead of exactly 1 because it averages many trees and rare spikes are difficult to capture. This behavior is normal in zero‑inflated retail data.

**2. Over‑prediction of zeros (true = 0, pred ≈ 0.63–0.86)**
These errors appear when recent rolling averages or lag features suggest demand, but the SKU happens not to sell on that specific day. RF predicts a small positive value because recent behavior indicates the SKU should have sold something. Retail noise makes these cases unavoidable.

**Error Distribution**
**mean error: 0.024**

**median error: 0.026*

max error: 1.0 (missing a rare 1‑unit sale)

**Overall, RF handles zero‑inflation extremely well and only struggles with rare positive events — a known limitation that XGBoost will address more effectively.**

# **Save RF Baseline Model**

In [5]:
# Notebook 3 — Cell 5: Save RF Baseline Model

import joblib

model_path = "models/random_forest_baseline.pkl"
joblib.dump(rf, model_path)

print(f"Saved Random Forest baseline model to: {model_path}")


Saved Random Forest baseline model to: models/random_forest_baseline.pkl
